# Day 2 — Types and values

> ⚠️ **Why this matters.** Half of all Python bugs are type confusion: you thought it was a string, it was a number; you thought it was a list, it was `None`. Type hints make those bugs visible before you run the code. Today you learn the types and start typing everything, like a senior engineer would.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/02-types-and-values.ipynb)

## What you'll do today

**Time:** 90 minutes.

By the end:

- [ ] You can list Python's core built-in types and what each is for
- [ ] You can use type hints fluently — every variable, every parameter, every return
- [ ] You understand `Optional[X]` / `X | None`
- [ ] You've run mypy against typed code and broken it on purpose
- [ ] You know when types are checked (statically) vs not (at runtime)

## The mental model

Every value in Python has a **type**. The type determines what you can do with it.

```
42        → int      (can do math)
3.14      → float    (can do math, imprecise)
"hello"   → str      (can slice, upper, lower)
True      → bool     (can do logic)
[1, 2, 3] → list     (can iterate, append)
{1, 2, 3} → set      (fast 'in' lookup)
{"a": 1}  → dict     (key-value)
None      → NoneType (the absence of a value)
```

**Python is dynamically typed** — you don't declare types up front. **Python supports type hints** — you can declare them anyway, for documentation and bug-catching. Modern Python code does declare them.

> 💡 **In the wild:** Every Python codebase at Anthropic, Stripe, Meta, Google, Microsoft — all heavily typed. `mypy --strict` is the bar. Code without types is treated as legacy.

## 1. The built-in types

`type()` tells you what something is.

In [1]:
type(42)

int

In [2]:
type("hello")

str

In [3]:
type([1, 2, 3])

list

In [4]:
type(None)

NoneType

## 2. Type hints — your code's self-documentation

A type hint after `:` on a variable declares what type it should be.

In [5]:
name: str = "Prince"
age: int = 16
height_m: float = 1.72
is_student: bool = True
favorite_numbers: list[int] = [7, 13, 42]

Type hints are **not enforced at runtime**. Python won't crash if you assign a string to a variable hinted as `int`. So why bother?

1. **Tools check them.** `mypy` (and your IDE) catches mistakes before you run.
2. **Self-documentation.** Anyone reading the code (including you, in 3 months) knows what's expected.
3. **Autocomplete.** Your editor knows what methods are available because it knows the type.

Run this — it doesn't crash, even though we lied about the type:

In [6]:
x: int = "not actually an int"   # mypy would complain, runtime doesn't care
print("actual type:", type(x))

actual type: <class 'str'>


> ⚠️ **Type hints are honor-system at runtime.** Run mypy as part of your workflow or they're decoration. We'll set up `mypy --strict` next.

## 3. Collections — list, tuple, set, dict

Four ways to hold multiple values. Each optimized for different patterns.

| Type | Syntax | Mutable? | Ordered? | Use when |
|------|--------|----------|----------|----------|
| `list` | `[1, 2, 3]` | yes | yes | You'll append and iterate |
| `tuple` | `(1, 2, 3)` | no | yes | Fixed group of related values |
| `set` | `{1, 2, 3}` | yes | no | You need fast `in` checks, unique values |
| `dict` | `{"a": 1}` | yes | yes (since 3.7) | Key-value mapping |

In [7]:
numbers: list[int] = [7, 13, 42]
numbers.append(99)
numbers

[7, 13, 42, 99]

In [8]:
unique: set[int] = {1, 2, 2, 3, 3, 4}   # duplicates dropped automatically
unique

{1, 2, 3, 4}

In [9]:
3 in unique     # set lookup is O(1) — instant even for millions of items

True

In [10]:
user: dict[str, int | str] = {"name": "Prince", "age": 16}
user["age"]

16

## 4. `None` and `Optional` / `| None`

`None` is the absence of a value. Different from `0`, `""`, `False`, `[]` — those are *values*. `None` is *no value*.

When something might be `None`, the type is `X | None` (modern) or `Optional[X]` (older). Both mean the same thing.

In [11]:
def greet(name: str | None) -> str:
    if name is None:
        return "Hello, stranger"
    return f"Hello, {name}"

greet("Prince")

'Hello, Prince'

In [12]:
greet(None)

'Hello, stranger'

> 💡 **Always check with `is None`, not `== None`.** The `is` operator checks identity (same object); `== None` calls `.__eq__()` which can be overridden in weird ways. `is None` is the idiom.

## 5. Conversions — when types meet

Python doesn't auto-convert (unlike JavaScript). You have to be explicit.

| From | To | Function |
|------|----|----------|
| any | str | `str(x)` |
| str | int | `int("42")` |
| str | float | `float("3.14")` |
| int/float | bool | `bool(x)` — 0/0.0/empty → False, else True |
| list | tuple / set | `tuple(xs)` / `set(xs)` |

In [13]:
age = 16
"I am " + str(age) + " years old."   # must convert int to str to concatenate

'I am 16 years old.'

In [14]:
user_input = "42"            # input() always returns str
doubled = int(user_input) * 2
doubled

84

## 6. Truthiness — when types are used as booleans

Python has a concept of "truthy" and "falsy." In an `if` or `while`, certain values count as `False` even when they're not actually `False`.

In [15]:
for value in [0, 1, "", "hello", [], [0], None]:
    print(f"{value!r} → {bool(value)}")

0 → False
1 → True
'' → False
'hello' → True
[] → False
[0] → True
None → False


**Falsy:** `False`, `None`, `0`, `0.0`, `""`, `[]`, `{}`, `set()`.

**Everything else** is truthy. So `if name:` works, but be careful — it'll also be False if `name == ""`. If you mean "name is None", say `if name is None:`.

## 7. Mini-exercise — write a typed function

Below, define a function `format_user`:

- Takes two parameters: `name: str` and `age: int | None`.
- Returns a `str`.
- If `age` is `None`, returns `"<name> (age unknown)"`.
- Otherwise returns `"<name>, <age>"`.

Examples:

```python
format_user("Prince", 16)    # → 'Prince, 16'
format_user("Alice", None)   # → 'Alice (age unknown)'
```

In [ ]:
# Your function here

def format_user(...) -> ...:
    ...

# Tests
assert format_user("Prince", 16) == "Prince, 16"
assert format_user("Alice", None) == "Alice (age unknown)"
print("All tests passed!")

<details>
<summary>Solution (try first!)</summary>

```python
def format_user(name: str, age: int | None) -> str:
    if age is None:
        return f"{name} (age unknown)"
    return f"{name}, {age}"
```
</details>

## 8. Now run mypy on a real file

Outside this notebook, in `~/prince/scratch/python-warmup/`:

```bash
cat > types_demo.py << 'EOF'
def add(a: int, b: int) -> int:
    return a + b

result = add("hello", 5)   # bug — mypy will catch it
print(result)
EOF

uv run mypy types_demo.py
```

Expected output:

```
types_demo.py:4: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
Found 1 error in 1 file (checked 1 source file)
```

mypy caught a bug **before** you ran the code. That's the magic.

Fix it — change `"hello"` to `3`. Re-run mypy:

```
Success: no issues found in 1 source file
```

Then run the file:

```bash
uv run python types_demo.py
# 8
```

> 🎯 **Make this a habit.** From now on, before you commit, you run: `uv run ruff format .`, `uv run ruff check .`, `uv run mypy .`. Three commands. They catch 80% of bugs before they exist.

## Connect to the project

> 🎯 **Connects to the project:** Every function in `pomo` will have type hints — every parameter, every return. The `Session` data class will be typed (`started_at: datetime`, `duration_min: int`, `tag: str | None`). The grading rubric requires `mypy --strict` passing. So today's lesson is the foundation for the Phase 1 capstone, not optional polish.

## Self-check

<details>
<summary>1. What's the difference between <code>list</code> and <code>tuple</code>?</summary>

Both are ordered sequences. `list` is mutable (you can add/remove/change). `tuple` is immutable (fixed once created). Use tuple for fixed, related groups; list for collections you'll modify.
</details>

<details>
<summary>2. Why are type hints not enforced at runtime?</summary>

Python prioritizes flexibility — type hints are a layer on top, optional. Enforcement is done by tools (mypy, your IDE) statically. This means types catch bugs at edit/check time, not at run time. It also means you have to actually run mypy for them to matter.
</details>

<details>
<summary>3. What does <code>list[int] | None</code> mean as a type?</summary>

Either a list of integers, OR `None`. Useful when a function might "return the list of results, or nothing."
</details>

<details>
<summary>4. Why use <code>is None</code> instead of <code>== None</code>?</summary>

`is` checks identity ("same object in memory"). `==` calls `.__eq__()`, which a custom class can override to lie. `None` is a singleton, so `is None` is unambiguous, fast, and idiomatic.
</details>

<details>
<summary>5. What's "truthy" mean? Give 3 falsy values that aren't <code>False</code>.</summary>

Truthy = a value that evaluates to `True` in an `if`/`while`. Falsy values include: `False`, `None`, `0`, `0.0`, `""` (empty string), `[]` (empty list), `{}` (empty dict), `set()` (empty set).
</details>

## What's next

Tomorrow: **control flow** — `if`, `for`, `while`, comprehensions. How you make code do different things in different situations.